<img height="100" src="https://i.postimg.cc/gjptBxF4/logo-gas-removebg-preview.png" width="250"/>

# Sensitivity Analysis - Morris
> analysis performed at : /util/SensitivityAnalysis.py

Article: <br>
Paleari, L., Movedi, E., Zoli, M., Burato, A., Cecconi, I., Errahouly, J., Pecollo, E., Sorvillo, C., & Confalonieri, R. (2021). Sensitivity analysis using Morris: Just screening or an effective ranking method?. Ecological Modelling. https://doi.org/10.1016/j.ecolmodel.2021.109648.

In [1]:
import pandas as pd
import os
from glob import glob
from warnings import filterwarnings

filterwarnings("ignore")

# Setting the path to the results folder
path_to_results = os.path.join(os.getcwd(), "output", "Sensitivity Analysis", "*.csv")
files = glob(path_to_results)

In [2]:
df = pd.read_csv(files[0], sep=",", header=0, skiprows=0)
df.head()

,parameter,mu_star,sigma,mu,point_id,cluster_id,latitude,longitude
0,AMAXTB000,5.476600e+03,3.587252e+04,5.453022e+03,101d9a64,0,40.75,111.75
1,AMAXTB014,1.728593e+05,5.061524e+05,1.640947e+05,101d9a64,0,40.75,111.75
2,AMAXTB082,1.210233e+06,2.438904e+06,1.210233e+06,101d9a64,0,40.75,111.75
3,AMAXTB200,4.213418e+05,8.909277e+05,4.213418e+05,101d9a64,0,40.75,111.75
4,TMPFTB000,0.000000e+00,0.000000e+00,0.000000e+00,101d9a64,0,40.75,111.75


In [3]:
# Creating a method to automatize .csv
def merge_csv(files):
    """
    The function reads multiple csv files, extracts location, year, and day from the file names,
    :param files: A list of paths to csv files.
    :return: A list of dataframes containing the processed data.
    """
    dataframes = []

    # Looping through all files
    for file in files:
        # Reading the csv file
        df = pd.read_csv(file, sep=",", header=0, skiprows=0)

        dataframes.append(df)

    return dataframes

# Creating a list of dataframes
dfs = merge_csv(files)

In [4]:
# Concatenating all dataframes into a single one
df_combined = pd.concat(dfs, ignore_index=True).round(2)
df_combined.head(), df_combined.tail()

(   parameter     mu_star       sigma          mu  point_id  cluster_id  \
 0  AMAXTB000     5476.60    35872.52     5453.02  101d9a64           0   
 1  AMAXTB014   172859.34   506152.43   164094.71  101d9a64           0   
 2  AMAXTB082  1210233.11  2438904.07  1210233.11  101d9a64           0   
 3  AMAXTB200   421341.84   890927.72   421341.84  101d9a64           0   
 4  TMPFTB000        0.00        0.00        0.00  101d9a64           0   
 
    latitude  longitude  
 0     40.75     111.75  
 1     40.75     111.75  
 2     40.75     111.75  
 3     40.75     111.75  
 4     40.75     111.75  ,
       parameter    mu_star       sigma         mu  point_id  cluster_id  \
 12745      TDWI  472440.68  1335579.37   19916.23  fc37a368           4   
 12746    TEFFMX       0.00        0.00       0.00  fc37a368           4   
 12747     TSUM1  423858.91   946027.47 -185988.24  fc37a368           4   
 12748     TSUM2   72269.27   202141.44  -35319.47  fc37a368           4   
 12749    T

In [5]:
# Grouping by cluster_id and parameter, then calculating the mean of mu_star and sigma
df_combined = df_combined.groupby(['cluster_id','parameter'])[['mu_star', 'sigma']].mean().sort_values(by='mu_star', ascending=False)

df_combined.reset_index(inplace=True); df_combined

,cluster_id,parameter,mu_star,sigma
0,2,AMAXTB082,2.194923e+06,3.228333e+06
1,3,TMPFTB020,1.832878e+06,2.590381e+06
2,2,TMPFTB020,1.801621e+06,2.528722e+06
3,2,TDWI,1.658979e+06,3.070620e+06
4,1,TMPFTB020,1.618191e+06,2.891327e+06
...,...,...,...,...
250,4,TBASEM,0.000000e+00,0.000000e+00
251,4,TEFFMX,0.000000e+00,0.000000e+00
252,4,TMPFTB008,0.000000e+00,0.000000e+00
253,4,TMPFTB000,0.000000e+00,0.000000e+00


In [ ]:
# Variables importance plot
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="white", context='paper', font_scale=1.2)
for cluster in df_combined['cluster_id'].unique():
    df_cluster = df_combined[df_combined['cluster_id'] == cluster]

    plt.figure(figsize=(10, 6))
    sns.barplot(x='mu_star', y='parameter', data=df_cluster, palette='viridis')
    plt.title(f'Variable Importance for Cluster ID: {cluster}')
    plt.xlabel('Mu Star')
    plt.ylabel('Parameter')
    plt.tight_layout()
    plt.show()

In [6]:
# List of zeros mu_star parameters
for cluster in df_combined['cluster_id'].unique():
    df_cluster = df_combined[df_combined['cluster_id'] == cluster]
    zero_mu_star_params = df_cluster[df_cluster['mu_star'] == 0]['parameter'].tolist()
    print(f"Cluster ID: {cluster} - Parameters with zero mu_star: {zero_mu_star_params}")

Cluster ID: 2 - Parameters with zero mu_star: ['EFFTB000', 'EFFTB040', 'TBASEM', 'TEFFMX', 'SLATB', 'TSUMEM']
Cluster ID: 3 - Parameters with zero mu_star: ['SLATB', 'EFFTB040', 'EFFTB000', 'TEFFMX', 'TBASEM', 'TMPFTB045', 'TSUMEM']
Cluster ID: 1 - Parameters with zero mu_star: ['TSUMEM', 'EFFTB000', 'EFFTB040', 'SLATB', 'TBASEM', 'TEFFMX']
Cluster ID: 0 - Parameters with zero mu_star: ['TSUMEM', 'TBASEM', 'TEFFMX', 'EFFTB040', 'EFFTB000', 'SLATB']
Cluster ID: 4 - Parameters with zero mu_star: ['EFFTB040', 'EFFTB000', 'SLATB', 'TBASEM', 'TEFFMX', 'TMPFTB008', 'TMPFTB000', 'TSUMEM']


In [7]:
# Parameter ranking by cluster
results = []
for cluster in df_combined['cluster_id'].unique():
    df_cluster = df_combined[(df_combined['cluster_id'] == cluster) & (df_combined['mu_star'] > 0)].copy()
    df_cluster = df_cluster.sort_values(by='mu_star', ascending=False).reset_index(drop=True)
    df_cluster['Rank'] = df_cluster.index + 1

    cluster_ranking = {
        'cluster_id': cluster,
        'ranking': df_cluster[['Rank', 'parameter', 'mu_star', 'sigma']].to_dict('records')
    }
    results.append(cluster_ranking)

results

[{'cluster_id': np.int64(2),
  'ranking': [{'Rank': 1,
    'parameter': 'AMAXTB082',
    'mu_star': 2194922.5846,
    'sigma': 3228332.713},
   {'Rank': 2,
    'parameter': 'TMPFTB020',
    'mu_star': 1801620.7136000001,
    'sigma': 2528721.7814},
   {'Rank': 3,
    'parameter': 'TDWI',
    'mu_star': 1658978.5244,
    'sigma': 3070620.4526},
   {'Rank': 4,
    'parameter': 'TMPFTB035',
    'mu_star': 1521705.55,
    'sigma': 2252239.0653999997},
   {'Rank': 5,
    'parameter': 'SLATB100',
    'mu_star': 1351036.3322,
    'sigma': 2351473.1188},
   {'Rank': 6,
    'parameter': 'SPAN',
    'mu_star': 1283945.365,
    'sigma': 2362332.0892},
   {'Rank': 7,
    'parameter': 'TSUM1',
    'mu_star': 1211753.148,
    'sigma': 2695383.83},
   {'Rank': 8,
    'parameter': 'DVSEND',
    'mu_star': 955486.6082000001,
    'sigma': 2955935.5112},
   {'Rank': 9,
    'parameter': 'AMAXTB200',
    'mu_star': 940056.5401999999,
    'sigma': 1582464.0916},
   {'Rank': 10,
    'parameter': 'DVSI',
    

In [8]:
# Lista simplificada dos top parâmetros por cluster
for cluster_data in results:
    cluster_id = cluster_data['cluster_id']
    top_params = [item['parameter'] for item in cluster_data['ranking'][:44]]
    print(f"Cluster {cluster_id}: {top_params}")

Cluster 2: ['AMAXTB082', 'TMPFTB020', 'TDWI', 'TMPFTB035', 'SLATB100', 'SPAN', 'TSUM1', 'DVSEND', 'AMAXTB200', 'DVSI', 'CVO', 'SLATB064', 'DLC', 'RFSETB200', 'RMO', 'IDSL', 'RMS', 'SLATB029', 'AMAXTB014', 'TBASE', 'RRI', 'RFSETB000', 'TSUM2', 'SLATB000', 'RML', 'RDMCR', 'CFET', 'CVS', 'SLATB021', 'RDI', 'RMR', 'RGRLAI', 'CVL', 'DLO', 'DEPNR', 'PERDL', 'IOX', 'IAIRDU', 'SLATB200', 'AMAXTB000', 'Q10', 'CVR', 'TMPFTB008', 'TMPFTB045']
Cluster 3: ['TMPFTB020', 'AMAXTB082', 'TSUM1', 'TDWI', 'DVSI', 'SPAN', 'SLATB100', 'SLATB064', 'AMAXTB200', 'DVSEND', 'CVO', 'TBASE', 'SLATB029', 'AMAXTB014', 'TSUM2', 'DLC', 'TMPFTB035', 'RMS', 'RFSETB200', 'IDSL', 'RMO', 'RFSETB000', 'TMPFTB008', 'SLATB000', 'RML', 'RMR', 'DLO', 'CVS', 'Q10', 'SLATB021', 'RRI', 'CVL', 'RGRLAI', 'SLATB200', 'CFET', 'AMAXTB000', 'RDI', 'DEPNR', 'CVR', 'RDMCR', 'PERDL', 'TMPFTB000', 'IAIRDU', 'IOX']
Cluster 1: ['TMPFTB020', 'AMAXTB082', 'TSUM1', 'TDWI', 'SPAN', 'DVSI', 'SLATB100', 'TMPFTB035', 'SLATB064', 'AMAXTB200', 'DVSEND

In [9]:
# del dfs, files, path_to_results

# Saving SA results
path_to_results = os.path.join(os.getcwd(), "output", "Sensitivity Analysis", "SA_result.xlsx")
df_combined.to_excel(path_to_results, index=True, header=True)